# Fine-tune LFM2.5-VL-1.6B on Multimodal-Mind2Web

Trains a web navigation agent on screenshot + task → next action prediction.  
Target hardware: RTX 4060 8GB VRAM.  
Output: LoRA adapter + GGUF export for llama.cpp CPU inference.

## 1. Install dependencies

In [ ]:
# Install Unsloth (nightly for latest SigLIP2 fixes) + deps
!uv pip install --upgrade --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!uv pip install --upgrade "transformers>=5.1" datasets trl bitsandbytes accelerate sentencepiece pillow

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"bf16 support: {torch.cuda.is_bf16_supported()}")

## 2. Load and explore the dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("osunlp/Multimodal-Mind2Web", split="train")
print(f"Training samples: {len(dataset)}")
print(f"Features: {list(dataset.features.keys())}")

In [ ]:
# Inspect a single sample
sample = dataset[0]

print("=== Task ===")
print(sample["confirmed_task"])

print("\n=== Operation ===")
print(sample["operation"])

print("\n=== Value ===")
print(sample.get("value", ""))

print("\n=== Target action ===")
print(sample["target_action_reprs"])

print("\n=== Action history (last 3) ===")
history = sample.get("action_reprs", [])
for h in history[-3:]:
    print(" -", h)

print(f"\n=== Candidates ===")
print(f"Positive: {len(sample['pos_candidates'])}")
print(f"Negative: {len(sample['neg_candidates'])}")

print("\n=== Screenshot ===")
img = sample["screenshot"]
print(f"Type: {type(img)}, Size: {img.size}")
display(img)

## 3. Data conversion

Each sample → Unsloth vision conversation format.

**User turn:** screenshot image + task + action history + shuffled labeled candidates  
**Assistant turn:** `ACTION: ... ELEMENT: [X] VALUE: ...` (structured, parseable)

In [ ]:
import random
from PIL import Image

SYSTEM_PROMPT = (
    "You are a web navigation agent. Given a screenshot of a webpage, a task description, "
    "prior action history, and a list of candidate DOM elements, predict the next action.\n"
    "Respond ONLY in this exact format (no prose):\n"
    "ACTION: <CLICK|TYPE|SELECT>\n"
    "ELEMENT: [<letter>]\n"
    "VALUE: <text or option to enter, or NONE>"
)

LETTER_LABELS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
MAX_CANDIDATES = 10   # cap to keep context short
MAX_IMG_SIZE   = 512  # max dimension (pixels)


def resize_image(img: Image.Image, max_size: int = MAX_IMG_SIZE) -> Image.Image:
    """Resize preserving aspect ratio so neither dimension exceeds max_size."""
    w, h = img.size
    if w <= max_size and h <= max_size:
        return img
    scale = max_size / max(w, h)
    return img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)


def candidate_text(c) -> str:
    """Return a compact text description of a candidate element."""
    if isinstance(c, dict):
        tag    = c.get("tag", "")
        text   = c.get("text", "").strip()
        attrs  = c.get("attributes", {})
        aria   = attrs.get("aria-label", "") if isinstance(attrs, dict) else ""
        parts  = [p for p in [tag, text or aria] if p]
        return " | ".join(parts) if parts else str(c)
    return str(c)


def convert_sample(sample: dict) -> dict | None:
    """
    Convert a Mind2Web sample to Unsloth vision conversation format.
    Returns None if the sample is unusable (missing screenshot / candidates).
    """
    img = sample.get("screenshot")
    if img is None:
        return None

    pos_candidates = sample.get("pos_candidates") or []
    neg_candidates = sample.get("neg_candidates") or []

    if not pos_candidates:
        return None

    # Build candidate pool: 1 positive + up to (MAX_CANDIDATES-1) negatives
    pos_candidate = pos_candidates[0]   # use first positive
    neg_pool      = neg_candidates[: MAX_CANDIDATES - 1]
    pool          = [pos_candidate] + neg_pool
    random.shuffle(pool)

    correct_idx = pool.index(pos_candidate)
    correct_letter = LETTER_LABELS[correct_idx]

    candidate_lines = [
        f"[{LETTER_LABELS[i]}] {candidate_text(c)}" for i, c in enumerate(pool)
    ]

    # Action history (last 5 steps)
    history = sample.get("action_reprs") or []
    history_text = (
        "\n".join(f"  {i+1}. {h}" for i, h in enumerate(history[-5:]))
        if history else "  (none)"
    )

    user_text = (
        f"TASK: {sample['confirmed_task']}\n\n"
        f"PRIOR ACTIONS:\n{history_text}\n\n"
        f"CANDIDATE ELEMENTS:\n" + "\n".join(candidate_lines)
    )

    operation = (sample.get("operation") or "CLICK").upper()
    value     = (sample.get("value") or "").strip()

    assistant_text = (
        f"ACTION: {operation}\n"
        f"ELEMENT: [{correct_letter}]\n"
        f"VALUE: {value if value else 'NONE'}"
    )

    img_resized = resize_image(img.convert("RGB"))

    return {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,     # must be a string, not a list
            },
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img_resized},
                    {"type": "text",  "text": user_text},
                ],
            },
            {
                "role": "assistant",
                "content": assistant_text,
            },
        ]
    }


# Smoke-test conversion
converted = convert_sample(dataset[0])
print("System:\n", converted["messages"][0]["content"])
print("\nUser text:\n", converted["messages"][1]["content"][1]["text"])
print("\nAssistant:\n", converted["messages"][2]["content"])

## 4. Convert full dataset

In [ ]:
from tqdm.auto import tqdm

random.seed(42)

converted_data = []
skipped = 0

for sample in tqdm(dataset, desc="Converting samples"):
    result = convert_sample(sample)
    if result is None:
        skipped += 1
    else:
        converted_data.append(result)

print(f"Converted: {len(converted_data):,}  |  Skipped: {skipped}")

## 5. Load model + LoRA setup

In [ ]:
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    model_name       = "LiquidAI/LFM2.5-VL-1.6B",
    max_seq_length   = 32768,
    load_in_4bit     = False,   # full bf16 — A100 80GB has the headroom
    use_gradient_checkpointing = "unsloth",
)

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,   # fine-tune SigLIP2 too — 80GB allows it
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r           = 16,
    lora_alpha  = 16,
    lora_dropout = 0,
    bias        = "none",
    random_state = 42,
    use_rslora  = False,
)
model.print_trainable_parameters()

## 6. Training

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

USE_BF16 = torch.cuda.is_bf16_supported()
print(f"Training with {'bf16' if USE_BF16 else 'fp16'}")

# max_steps=500 for a quick run (~20-30 min on A100 SXM 80GB).
# For full epochs: remove max_steps and set num_train_epochs=3
# Full dataset at batch 4 * grad_accum 4 = effective batch 16 ≈ 486 steps/epoch.
MAX_STEPS = 500

trainer = SFTTrainer(
    model     = model,
    tokenizer = tokenizer,
    train_dataset    = converted_data,
    data_collator    = UnslothVisionDataCollator(model, tokenizer),
    args = SFTConfig(
        output_dir                  = "./lfm25-mind2web-lora",
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,   # effective batch size = 16
        gradient_checkpointing      = True,
        learning_rate               = 2e-4,
        lr_scheduler_type           = "cosine",
        warmup_steps                = 20,
        max_steps                   = MAX_STEPS,
        # num_train_epochs            = 3,      # uncomment for full epochs
        optim                       = "adamw_8bit",
        bf16                        = USE_BF16,
        fp16                        = not USE_BF16,
        logging_steps               = 10,
        save_steps                  = 100,
        save_total_limit            = 3,
        dataset_text_field          = "",      # vision dataset uses messages
        dataset_kwargs              = {"skip_prepare_dataset": True},
        remove_unused_columns       = False,
        seed                        = 42,
        report_to                   = "none",
    ),
)

trainer_stats = trainer.train()
print(trainer_stats)

## 7. Quick inference test

In [ ]:
FastVisionModel.for_inference(model)

# Grab a held-out sample (use a later index not seen during 500-step training)
test_sample = dataset[7000]
test_converted = convert_sample(test_sample)

# Ground truth
print("Ground truth:", test_converted["messages"][2]["content"])
print()

# Build the prompt without the assistant turn
messages_no_answer = test_converted["messages"][:2]

input_text = tokenizer.apply_chat_template(
    messages_no_answer,
    add_generation_prompt = True,
    tokenize = False,
)

# Extract image
test_image = test_converted["messages"][1]["content"][0]["image"]

inputs = tokenizer(
    images = test_image,
    text   = input_text,
    return_tensors = "pt",
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens = 64,
        temperature    = 0.0,
        do_sample      = False,
    )

# Decode only the generated tokens
generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
prediction = tokenizer.decode(generated_ids, skip_special_tokens=True)
print("Prediction:", prediction)

## 8. Save LoRA adapter

In [ ]:
ADAPTER_DIR = "lfm25-mind2web-lora"

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"LoRA adapter saved to ./{ADAPTER_DIR}")

## 9. Merge LoRA + export to GGUF

Merges weights, then quantizes to Q4_0 (smallest/fastest) and Q8_0 (higher quality) for llama.cpp.

In [ ]:
# Merge LoRA into base weights and save as full bf16 model
MERGED_DIR = "lfm25-mind2web-merged"

model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method = "merged_16bit",
)
print(f"Merged model saved to ./{MERGED_DIR}")

In [ ]:
# Export Q4_0 GGUF — smallest file, fastest CPU inference
model.save_pretrained_gguf(
    "lfm25-mind2web-q4",
    tokenizer,
    quantization_method = "q4_0",
)
print("Q4_0 GGUF saved to ./lfm25-mind2web-q4/")

In [ ]:
# Export Q8_0 GGUF — larger file, better accuracy
model.save_pretrained_gguf(
    "lfm25-mind2web-q8",
    tokenizer,
    quantization_method = "q8_0",
)
print("Q8_0 GGUF saved to ./lfm25-mind2web-q8/")

## 10. CPU inference with llama.cpp

### Build llama.cpp with vision support

```bash
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp
cmake -B build -DLLAMA_CURL=ON
cmake --build build --config Release -j$(nproc)
```

### Run inference (Q4_0)

LFM2.5-VL uses the `llava` vision architecture in llama.cpp.  
You'll need both the main GGUF and the mmproj (vision projector) GGUF that Unsloth exports.

```bash
# List exported files — look for *-mmproj-*.gguf
ls lfm25-mind2web-q4/

# Run a single inference
./build/bin/llama-llava-cli \
  --model      lfm25-mind2web-q4/model-q4_0.gguf \
  --mmproj     lfm25-mind2web-q4/mmproj-model-f16.gguf \
  --image      /path/to/screenshot.png \
  --prompt     "<|im_start|>system\nYou are a web navigation agent..." \
  --n-predict  64 \
  --threads    8
```

### Python wrapper (llama-cpp-python)

```bash
pip install llama-cpp-python
```

```python
from llama_cpp import Llama
from llama_cpp.llama_chat_format import Llava15ChatHandler

chat_handler = Llava15ChatHandler(
    clip_model_path="lfm25-mind2web-q4/mmproj-model-f16.gguf"
)

llm = Llama(
    model_path   = "lfm25-mind2web-q4/model-q4_0.gguf",
    chat_handler = chat_handler,
    n_ctx        = 2048,
    n_threads    = 8,
    verbose      = False,
)

import base64

def image_to_data_uri(path: str) -> str:
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode()
    return f"data:image/png;base64,{data}"

response = llm.create_chat_completion(
    messages=[
        {
            "role": "system",
            "content": (
                "You are a web navigation agent. Given a screenshot of a webpage, "
                "a task description, prior action history, and a list of candidate DOM elements, "
                "predict the next action.\n"
                "Respond ONLY in this exact format (no prose):\n"
                "ACTION: <CLICK|TYPE|SELECT>\n"
                "ELEMENT: [<letter>]\n"
                "VALUE: <text or option to enter, or NONE>"
            ),
        },
        {
            "role": "user",
            "content": [
                {"type": "image_url",
                 "image_url": {"url": image_to_data_uri("screenshot.png")}},
                {"type": "text",
                 "text": "TASK: ...\nPRIOR ACTIONS: ...\nCANDIDATE ELEMENTS: ..."},
            ],
        },
    ],
    max_tokens = 64,
    temperature = 0.0,
)

print(response["choices"][0]["message"]["content"])
```

### Expected output format

```
ACTION: CLICK
ELEMENT: [C]
VALUE: NONE
```

Parse with:
```python
import re

def parse_action(text: str) -> dict:
    action  = re.search(r"ACTION:\s*(\w+)", text)
    element = re.search(r"ELEMENT:\s*\[([A-Z])\]", text)
    value   = re.search(r"VALUE:\s*(.+)", text)
    return {
        "action":  action.group(1) if action else None,
        "element": element.group(1) if element else None,
        "value":   value.group(1).strip() if value else None,
    }
```